
###### 18_nlp_model_evaluation

###### Purpose
Evaluate trained Support Ticket NLP models, select the best model using f1_macro, and save the best model metadata to Unity Catalog for downstream registration.

###### Input
- Read model_training_results,  sort by best metric and get the best model info

######  Output
- write the best model info results to the table dbw_agentic_ai_dev.support_ticket_ai.best_model_info

######  Architecture

```text

model_training_results
      ↓
Sort by f1_macro DESC
      ↓
Tie-break by accuracy DESC
      ↓
Select best model row
      ↓
Create best_model_info
      ↓
Write to Unity Catalog Delta table

```


###### Skills Covered

Delta Lake and Unity Catalog.

###### Section : Get the best model info

In [0]:

#Read model_training_results
results_df = spark.table("dbw_agentic_ai_dev.support_ticket_ai.model_training_results")

#Sort by best metric
display(results_df.orderBy("f1_macro", ascending=False))

#Pick best row 
best_row = (
    results_df
    .orderBy(
        results_df["f1_macro"].desc(),
        results_df["accuracy"].desc()
    )
    .limit(1)
    .collect()[0]
)

best_model_info = [{
    "best_run_id": best_row["run_id"],
    "best_model_name": best_row["model"],
    "best_vectorizer": best_row["vectorizer"],
    "best_vocabulary_size": int(best_row["vocabulary_size"]),
    "best_accuracy": float(best_row["accuracy"]),
    "best_precision_macro": float(best_row["precision_macro"]),
    "best_recall_macro": float(best_row["recall_macro"]),
    "best_f1_macro": float(best_row["f1_macro"])
}]

#Create best_model_info
best_model_df = spark.createDataFrame(best_model_info)

#Write to the delta table
best_model_df.write.format("delta").mode("overwrite").saveAsTable("dbw_agentic_ai_dev.support_ticket_ai.best_model_info")

display(best_model_df)


###### Notebook Summary

- Read model_training_results
- Sort by best metric and pick best row 
- Create best_model_info and Write to the delta table - dbw_agentic_ai_dev.support_ticket_ai.best_model_info


######  Key Learnings

- For binary classification, ROC-AUC is commonly used when comparing churn/no-churn models.
- For multi-class classification, f1_macro is better because it gives equal importance to every class.
- Accuracy alone can hide poor performance on smaller or weaker classes.


###### Next Notebook

19_nlp_mlflow_tracking

This notebook will validate the MLflow experiment and artifacts.